In [3]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5)

print(qc)

     
q_0: 
     
q_1: 
     
q_2: 
     
q_3: 
     
q_4: 
     


In [4]:
qc.cx(0, 3)   # A -> Sum
qc.cx(1, 3)   # B -> Sum
qc.cx(2, 3)   # Cin -> Sum

print(qc)

                    
q_0: ──■────────────
       │            
q_1: ──┼────■───────
       │    │       
q_2: ──┼────┼────■──
     ┌─┴─┐┌─┴─┐┌─┴─┐
q_3: ┤ X ├┤ X ├┤ X ├
     └───┘└───┘└───┘
q_4: ───────────────
                    


In [5]:
qc.ccx(0, 1, 4)   # A AND B -> Carry
qc.ccx(0, 2, 4)   # A AND Cin -> Carry
qc.ccx(1, 2, 4)   # B AND Cin -> Carry

print(qc)

                                   
q_0: ──■──────────────■────■───────
       │              │    │       
q_1: ──┼────■─────────■────┼────■──
       │    │         │    │    │  
q_2: ──┼────┼────■────┼────■────■──
     ┌─┴─┐┌─┴─┐┌─┴─┐  │    │    │  
q_3: ┤ X ├┤ X ├┤ X ├──┼────┼────┼──
     └───┘└───┘└───┘┌─┴─┐┌─┴─┐┌─┴─┐
q_4: ───────────────┤ X ├┤ X ├┤ X ├
                    └───┘└───┘└───┘


In [6]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(5, 5)

# Input: A=1, B=1, Cin=0
qc.x(0)
qc.x(1)

# Sum
qc.cx(0, 3)
qc.cx(1, 3)
qc.cx(2, 3)

# Carry
qc.ccx(0, 1, 4)
qc.ccx(0, 2, 4)
qc.ccx(1, 2, 4)

# Measurement
qc.measure([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])

print(qc)

     ┌───┐                                 ┌─┐         
q_0: ┤ X ├──■──────────────■────■──────────┤M├─────────
     ├───┤  │              │    │          └╥┘┌─┐      
q_1: ┤ X ├──┼────■─────────■────┼───────■───╫─┤M├──────
     └───┘  │    │         │    │       │   ║ └╥┘┌─┐   
q_2: ───────┼────┼────■────┼────■───────■───╫──╫─┤M├───
          ┌─┴─┐┌─┴─┐┌─┴─┐  │    │  ┌─┐  │   ║  ║ └╥┘   
q_3: ─────┤ X ├┤ X ├┤ X ├──┼────┼──┤M├──┼───╫──╫──╫────
          └───┘└───┘└───┘┌─┴─┐┌─┴─┐└╥┘┌─┴─┐ ║  ║  ║ ┌─┐
q_4: ────────────────────┤ X ├┤ X ├─╫─┤ X ├─╫──╫──╫─┤M├
                         └───┘└───┘ ║ └───┘ ║  ║  ║ └╥┘
c: 5/═══════════════════════════════╩═══════╩══╩══╩══╩═
                                    3       0  1  2  4 


In [7]:
from qiskit_aer import AerSimulator

simulator = AerSimulator()

result = simulator.run(qc, shots=1000).result()

counts = result.get_counts()

print(counts)

{'10011': 1000}


In [8]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def full_adder_circuit(a, b, cin):
    qc = QuantumCircuit(5, 5)

    # Prepare inputs
    if a == 1:
        qc.x(0)

    if b == 1:
        qc.x(1)

    if cin == 1:
        qc.x(2)

    # Sum = A XOR B XOR Cin
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(2, 3)

    # Carry = AB XOR ACin XOR BCin
    qc.ccx(0, 1, 4)
    qc.ccx(0, 2, 4)
    qc.ccx(1, 2, 4)

    # Measurement
    qc.measure([0, 1, 2, 3, 4], [0, 1, 2, 3, 4])

    return qc

In [9]:
test_cases = [
    (0, 0, 0),
    (0, 0, 1),
    (0, 1, 0),
    (0, 1, 1),
    (1, 0, 0),
    (1, 0, 1),
    (1, 1, 0),
    (1, 1, 1)
]

print("A B Cin | Sum Carry")
print("--------------------")

for a, b, cin in test_cases:

    qc = full_adder_circuit(a, b, cin)

    result = simulator.run(qc, shots=100).result()
    counts = result.get_counts()

    measured = list(counts.keys())[0]

    # Qiskit bit order: c4 c3 c2 c1 c0
    carry = int(measured[0])
    sum_bit = int(measured[1])

    print(f"{a} {b}  {cin}  |  {sum_bit}    {carry}")

A B Cin | Sum Carry
--------------------
0 0  0  |  0    0
0 0  1  |  1    0
0 1  0  |  1    0
0 1  1  |  0    1
1 0  0  |  1    0
1 0  1  |  0    1
1 1  0  |  0    1
1 1  1  |  1    1


In [10]:
qc = full_adder_circuit(1, 1, 0)

print("Number of qubits:", qc.num_qubits)
print("Number of classical bits:", qc.num_clbits)
print("Circuit depth:", qc.depth())
print("Gate count:", qc.count_ops())

Number of qubits: 5
Number of classical bits: 5
Circuit depth: 7
Gate count: OrderedDict({'measure': 5, 'cx': 3, 'ccx': 3, 'x': 2})


In [13]:
qc.draw("mpl")
fig = qc.draw("mpl", fold=-1)
fig.savefig("../results/circuits/full_adder.png")